In [ ]:
import os
import logging
import argparse
import numpy as np
import pandas as pd
from datetime import timedelta, datetime
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import sys
sys.argv = ['']

In [ ]:
# count patients
def count_subdirectories(path):
    """Counts the number of subdirectories in a given path."""
    return sum(1 for entry in os.scandir(path) if entry.is_dir())

In [ ]:
directory_path = "/opt/data/commonfilesharePHI/ldiao/ckd_project/ckd_embeddings_m_full_v2"
num_subdirs = count_subdirectories(directory_path)
print(f"Number of subdirectories: {num_subdirs}")

In [ ]:
directory_path = "/opt/data/commonfilesharePHI/ldiao/ckd_project/ckd_embedding_full_icd_stage_filter"
num_subdirs = count_subdirectories(directory_path)
print(f"Number of subdirectories: {num_subdirs}")

In [ ]:
# count embeddings
import os

def count_files(path):
    """
    Counts all files within a given directory and its subdirectories.
    
    Args:
        path (str): The starting directory path.
    
    Returns:
        int: The total count of files.
    """
    total_files = 0
    for dirpath, dirnames, filenames in os.walk(path):
        total_files += len(filenames)
    return total_files

num_files = count_files(directory_path)
print(f"Total number of files: {num_files}")

In [ ]:
def clean_ckd_stage(value):
    try:
        return int(value)
    except ValueError:
        if isinstance(value, str) and value[0].isdigit():
            return int(value[0])
        else:
            return np.nan
            
def filter_patients_by_ckd_stage(df, ckd_stage_col, patient_id_col='PatientID'):
    initial_patients = df[patient_id_col].nunique()
    # Filter for visits where CKD stage is 3 or higher
    df_at_or_above_stage_3 = df[df[ckd_stage_col] >= 3]
    # Get unique PatientIDs from this filtered DataFrame
    patient_ids_to_keep = set(df_at_or_above_stage_3[patient_id_col].unique())
    
    patients_removed = initial_patients - len(patient_ids_to_keep)

    return patient_ids_to_keep        

def find_CKD_stage_progression(df):
    df_sorted = df.sort_values(by=['PatientID', 'EventDate_dt'])
    
    # difference in CKD_stage for each patient
    df_sorted['stage_diff'] = df_sorted.groupby('PatientID')['CKD_stage_clean'].diff()

    # filter where the stage difference is positive (i.e., increased)
    df_increased = df_sorted[df_sorted['stage_diff'] > 0].copy()

    # retreive previous CKD_stage for context
    df_increased['previous_CKD_stage'] = df_sorted.groupby('PatientID')['CKD_stage_clean'].shift(1)
    
    # rename relevant columns
    result = df_increased[['PatientID', 'EventDate_dt', 'previous_CKD_stage', 'CKD_stage_clean']]
    result.rename(columns={'CKD_stage_clean': 'new_CKD_stage'}, inplace=True)
    
    return result

def unique_patient_ckd_counts(df):
    # Select only the necessary columns and drop duplicate rows based on PatientID
    # to ensure each patient is counted only once for their CKD stage.
    unique_patients_ckd = df[['PatientID', 'CKD_stage_clean']].drop_duplicates(subset=['PatientID'])

    # Count the occurrences of each CKD stage among these unique patients
    ckd_stage_counts = unique_patients_ckd['CKD_stage_clean'].value_counts()

    return ckd_stage_counts.sort_index()

In [ ]:
## check embeddings

In [ ]:
embedding_size = "full/" # 10, 100, full
embedding_path =  "./../../../commonfilesharePHI/slee/ckd-optum/ckd_embeddings_" + embedding_size

In [ ]:
metadata = pd.read_csv(embedding_path +"patient_embedding_metadata.csv")

In [ ]:
metadata.head()

In [ ]:
metadata['CKD_stage_clean'] = metadata['CKD_stage'].apply(clean_ckd_stage)
metadata = metadata.sort_values(by=['PatientID', 'EventDate'])
metadata['CKD_stage_clean'] = metadata.groupby('PatientID')['CKD_stage_clean'].bfill().ffill()
metadata = metadata.dropna(subset=['CKD_stage_clean'])
metadata['CKD_stage_clean'] = metadata['CKD_stage_clean'].astype(int)
metadata['label'] = metadata['CKD_stage_clean'].apply(lambda x: 1 if x >= 4 else 0)

In [ ]:
metadata.CKD_stage.unique()

In [ ]:
metadata[metadata['CKD_stage'].isin(['3a', '3b'])].shape

In [ ]:
metadata.CKD_stage_clean.unique()

In [ ]:
len(metadata[metadata['CKD_stage_clean'].isin([3])].PatientID.unique())

In [ ]:
len(metadata[metadata['CKD_stage'].isin(['3a', '3b'])].PatientID.unique())

In [ ]:
len(set(metadata['PatientID'].unique()))

In [ ]:
metadata.shape

In [ ]:
df = metadata
df.shape
# check nan rows
# nan_mean = df.isnull().mean()
# nan_mean

In [ ]:
df_patients = filter_patients_by_ckd_stage(metadata, 'CKD_stage_clean')
df = df[df["PatientID"].isin(df_patients)].copy()

In [ ]:
# total number of patients after filtering
npatients = len(df['PatientID'].unique())
print(npatients)

In [ ]:
df.shape

In [ ]:
## check tab gen / patient subset all csv

In [ ]:
## check results/output
# check the original output files before filtering. with the raw data. check patientID's

In [ ]:
# compare progression to true values (pre filtering)
df1 = pd.read_csv("./365day_future_prediction_outputs_50/LSTM_365DayFutureTarget_detailed_outputs.csv")
df1.head()

In [ ]:
# compare progression to true values (post filtering)
df2 = pd.read_csv("./365day_future_prediction_outputs_50_filter_stage_3/LSTM_365DayFutureTarget_detailed_outputs.csv")
df2.head()

In [ ]:
len(set(df1['PatientID'].unique()))

In [ ]:
len(set(df2['PatientID'].unique()))

In [ ]:
len(set(df1['PatientID'].unique()) - set(df2['PatientID'].unique()))

In [ ]:
len(set(df2['PatientID'].unique()) - set(df1['PatientID'].unique()))